# 🧠 Local AI with Ollama & Agno — A Complete Tutorial

> **Welcome, student!** This notebook is your guided journey from zero to hero with local LLMs. We'll start simple — asking a model to summarize text or write a story — and gradually unlock more powerful tools: web search, code execution, vision models, and multi-agent systems.
>
> **Models we'll use:**
> - `deepseek-ocr:3b` — specialized for OCR & vision tasks
> - `qwen3.5:4b` — a strong general-purpose language model

---

## 🗺️ Tutorial Roadmap

| Chapter | Topic | Complexity |
|---------|-------|------------|
| 1 | Setup & Installation | 🟢 Beginner |
| 2 | Simple Tasks (Summarize, Story, Plan) | 🟢 Beginner |
| 3 | Vision & OCR (Image → Text) | 🟡 Intermediate |
| 4 | Adding Web Search Tool | 🟡 Intermediate |
| 5 | Code Execution Tool | 🟠 Advanced |
| 6 | Multiple LLMs in One Pipeline | 🟠 Advanced |
| 7 | Multi-Agent Systems | 🔴 Expert |

---

### 📌 Prerequisites

Before running this notebook, make sure you have:
1. **Ollama** installed → [https://ollama.com](https://ollama.com)
2. Models pulled:
   ```bash
   ollama pull qwen3.5:4b
   ollama pull deepseek-ocr:3b
   ```
3. Ollama running: `ollama serve`

Let's begin! 🚀

---
# Chapter 1 — ⚙️ Setup & Installation

First, let's install the required Python packages. **Agno** is a lightweight, powerful framework for building AI agents. It wraps your local Ollama models and gives you tools, memory, and multi-agent capabilities.

In [ ]:
# Install required packages
!pip install agno ollama requests pillow duckduckgo-search --quiet

print("✅ Packages installed successfully!")

In [ ]:
# Let's verify Ollama is running and our models are available
import subprocess
import json

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print("📦 Available Ollama Models:")
print(result.stdout)

if "qwen3.5" not in result.stdout:
    print("⚠️  qwen3.5:4b not found. Run: ollama pull qwen3.5:4b")
if "deepseek-ocr" not in result.stdout:
    print("⚠️  deepseek-ocr:3b not found. Run: ollama pull deepseek-ocr:3b")

In [ ]:
# Import core libraries we'll use throughout the tutorial
from agno.agent import Agent
from agno.models.ollama import Ollama
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.python import PythonTools

print("✅ Agno imported successfully!")

# Define our model names as constants — easy to change later
GENERAL_MODEL = "qwen3.5:4b"
VISION_MODEL  = "deepseek-ocr:3b"

print(f"🤖 General LLM  : {GENERAL_MODEL}")
print(f"👁️  Vision/OCR LLM: {VISION_MODEL}")

---
# Chapter 2 — 🟢 Simple Tasks with a Local LLM

Let's start with the fundamentals. We'll create a basic Agno `Agent` powered by `qwen3.5:4b` running locally via Ollama.

> **Concept:** An `Agent` in Agno is a conversational entity. You give it a model, an optional personality (`description`/`instructions`), and then you call `.run()` or `.print_response()` with a message.

## 2.1 — Text Summarization

In [ ]:
# --- CREATE A SUMMARIZER AGENT ---
# We give it a clear role through the `description` parameter.
# The model runs entirely on your machine — no API keys needed!

summarizer = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a concise summarizer. Extract the key points from any text in 3-5 bullet points.",
    markdown=True
)

# A long piece of text to summarize
article = """
The James Webb Space Telescope (JWST) has fundamentally changed our understanding of the early universe.
Launched on December 25, 2021, it can observe infrared light from objects more than 13 billion light-years away.
One of its most stunning achievements was capturing the Pillars of Creation in unprecedented detail,
revealing thousands of previously unseen stars. The telescope has also discovered galaxies that formed
just 300 million years after the Big Bang — far earlier than scientists predicted. Its data is reshaping
models of galaxy formation and challenging existing cosmological theories. Scientists worldwide now have
access to this data through public archives, democratizing space research like never before.
"""

print("📄 Summarizing article about JWST...\n")
summarizer.print_response(f"Summarize this article:\n{article}")

In [ ]:
# --- SECOND RUN: Summarize a different topic ---
# Notice we reuse the SAME agent — it's stateless by default (no memory between runs)

tech_article = """
Quantum computing represents a paradigm shift in computational power. Unlike classical computers that use
bits (0 or 1), quantum computers use qubits that can exist in superposition — being both 0 and 1 simultaneously.
Through quantum entanglement and interference, they can solve certain problems exponentially faster.
IBM's latest quantum processor, Heron, boasts 133 qubits with record-low error rates. Google's Sycamore
performed a specific calculation in 200 seconds that would take a classical supercomputer 10,000 years.
The most promising applications include drug discovery, cryptography, and optimization problems in logistics.
However, quantum computers are not universally faster — they excel at specific problem types only.
"""

print("💻 Summarizing article about Quantum Computing...\n")
summarizer.print_response(f"Summarize this article in bullet points:\n{tech_article}")

In [ ]:
# --- THIRD RUN: Summarize and extract action items ---
# We can ask the same agent to do a different summarization style!

meeting_notes = """
Q3 Product Planning Meeting — June 15, 2025
Attendees: Sarah (PM), Marcus (Dev Lead), Priya (Design), Tom (Marketing)

We discussed the upcoming v2.0 release. Marcus noted the authentication module is 80% done but
needs 2 more weeks. Priya showed the new onboarding flow mockups — everyone loved the dark mode.
Tom wants a landing page update before the launch. Sarah agreed to write product copy by June 22.
The team decided to delay the analytics dashboard to v2.1 to avoid scope creep.
Launch target is July 10. Marcus will handle deployment. Tom will prepare press kit by July 5.
Next meeting: June 22 at 2pm.
"""

print("📋 Extracting action items from meeting notes...\n")
summarizer.print_response(
    f"From these meeting notes, extract: 1) Key decisions, 2) Action items with owners, 3) Deadlines:\n{meeting_notes}"
)

## 2.2 — Creative Writing: Write a Story

> **Student note:** Notice how we change the `description` to completely transform the agent's personality. The same underlying model becomes a creative writer!

In [ ]:
# --- CREATIVE WRITER AGENT ---

storyteller = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a masterful short story writer. You write vivid, emotionally resonant stories "
                "with strong characters, unexpected twists, and beautiful prose. Keep stories under 300 words.",
    markdown=True
)

print("✍️  Story 1: The Last Lighthouse Keeper\n")
storyteller.print_response(
    "Write a short story about the last lighthouse keeper on Earth who discovers something unexpected in the fog."
)

In [ ]:
# --- SECOND STORY: Science Fiction genre ---

print("🚀 Story 2: First Contact\n")
storyteller.print_response(
    "Write a science fiction short story: humanity makes first contact with an alien civilization, "
    "but the aliens turn out to be incredibly shy. Include a surprising and heartwarming ending."
)

In [ ]:
# --- THIRD STORY: Collaborative continuation ---
# We can give the model a starting sentence and let it continue!

opening = "The old bookshop had been closed for thirty years when Mira found the key in her grandmother's attic."

print("📚 Story 3: The Magical Bookshop (continuation)\n")
storyteller.print_response(
    f"Continue this story into a complete short story with a magical twist:\n\n'{opening}'"
)

## 2.3 — Vacation Planning in Rome 🏛️

> **Student note:** Here we use the `instructions` parameter to give the agent detailed behavioral rules. Think of `description` as the agent's job title, and `instructions` as its operating manual.

In [ ]:
# --- TRAVEL PLANNER AGENT ---

travel_agent = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are an expert travel planner with deep knowledge of European cities, culture, and history.",
    instructions=[
        "Always structure plans with Day-by-Day itineraries",
        "Include morning, afternoon, and evening activities",
        "Mention specific restaurant and food recommendations",
        "Add practical tips like best time to visit landmarks, ticket booking advice",
        "Include budget estimates in EUR",
    ],
    markdown=True
)

print("🏛️  Planning a 3-day Rome vacation...\n")
travel_agent.print_response(
    "Plan a perfect 3-day vacation in Rome for a couple who love history, art, and authentic Italian food. "
    "They have a moderate budget (€100-150/day excluding accommodation)."
)

In [ ]:
# --- SECOND PLAN: Rome for foodies only ---

print("🍝 Planning a Rome food tour...\n")
travel_agent.print_response(
    "Create a 2-day Rome itinerary entirely focused on food experiences: "
    "markets, cooking classes, best trattorias, gelato tours, and wine bars. "
    "Make it feel like a food journalist's itinerary."
)

In [ ]:
# --- THIRD PLAN: Hidden gems in Rome ---

print("🗺️  Discovering Rome's hidden gems...\n")
travel_agent.print_response(
    "Suggest 10 off-the-beaten-path places in Rome that most tourists miss. "
    "Include: a secret neighborhood, hidden churches, underground sites, and local bars. "
    "Format as a discovery guide with why each place is special."
)

## 2.4 — Bonus Simple Tasks

Let's explore a few more unique applications of a local LLM — things you wouldn't typically think to try!

In [ ]:
# --- SOCRATIC TUTOR: Explain concepts by asking questions ---

tutor = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a Socratic tutor. Instead of explaining directly, guide the student "
                "to discover answers through carefully crafted questions. Then reveal the full explanation.",
    markdown=True
)

print("🎓 Socratic Teaching: Why does the sky appear blue?\n")
tutor.print_response("Teach me why the sky is blue using the Socratic method.")

In [ ]:
# --- DEBATE COACH: Argue both sides of a topic ---

debater = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a professional debate coach. Present compelling arguments for BOTH sides of any topic, "
                "then give a balanced conclusion. Label sections clearly as FOR and AGAINST.",
    markdown=True
)

print("⚖️  Debate: Should AI replace human creativity?\n")
debater.print_response("Present arguments for and against: 'AI will replace human creativity in art and music.'")

In [ ]:
# --- RECIPE INVENTOR: Create a recipe from random ingredients ---

chef = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a creative Michelin-star chef who invents elegant recipes from whatever ingredients are given. "
                "Include: dish name, ingredients with quantities, step-by-step method, and plating suggestion.",
    markdown=True
)

print("👨‍🍳 Inventing a recipe from random ingredients...\n")
chef.print_response(
    "Create an elegant dinner recipe using ONLY these ingredients: "
    "leftover rice, canned tuna, one lemon, eggs, parmesan, olive oil, and rosemary. "
    "Make it sound like it belongs in a fine dining restaurant."
)

---
# Chapter 3 — 👁️ Vision & OCR Tasks with `deepseek-ocr:3b`

Now we step into **multimodal** territory. The `deepseek-ocr:3b` model can process images and extract text from them.

> **Student note:** Vision models accept images as input alongside text. Agno makes this seamless — you just pass the image path or URL and the model does the rest. OCR (Optical Character Recognition) + scene understanding in one model!

In [ ]:
# --- SETUP: Import image handling utilities ---
from agno.agent import Agent
from agno.models.ollama import Ollama
from agno.media import Image
import urllib.request
import os

# Create a directory for our test images
os.makedirs("test_images", exist_ok=True)
print("📁 test_images/ directory created")

In [ ]:
# --- DOWNLOAD TEST IMAGES ---
# We'll use publicly available images for our tests

test_images = {
    "handwritten.png": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8e/Handwriting_of_Barack_Obama.jpg/400px-Handwriting_of_Barack_Obama.jpg",
    "street_sign.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Street_signs_in_New_York.jpg/320px-Street_signs_in_New_York.jpg",
    "document.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f8/Declaration_of_independence_1776_detail.jpg/400px-Declaration_of_independence_1776_detail.jpg",
}

for filename, url in test_images.items():
    path = f"test_images/{filename}"
    try:
        urllib.request.urlretrieve(url, path)
        print(f"✅ Downloaded: {filename}")
    except Exception as e:
        print(f"⚠️  Could not download {filename}: {e}")

print("\n💡 Tip: If downloads fail, place your own images in the test_images/ folder.")

In [ ]:
# --- CREATE VISION AGENT ---

vision_agent = Agent(
    model=Ollama(id=VISION_MODEL),
    description="You are an expert at analyzing images. You can read text (OCR), describe scenes, "
                "identify objects, and provide detailed image analysis.",
    markdown=True
)

print(f"👁️  Vision Agent created with model: {VISION_MODEL}")

In [ ]:
# --- OCR TASK 1: Read handwritten text ---

print("✍️  Task: Read handwritten text from image\n")

vision_agent.print_response(
    "Please read and transcribe ALL text visible in this handwritten image. "
    "Then describe the handwriting style.",
    images=[Image(filepath="test_images/handwritten.png")]
)

In [ ]:
# --- OCR TASK 2: Read street signs ---

print("🚦 Task: Read all text from street signs\n")

vision_agent.print_response(
    "Read every piece of text you can see in this image. "
    "List them clearly, and describe where each text appears in the image.",
    images=[Image(filepath="test_images/street_sign.jpg")]
)

In [ ]:
# --- OCR TASK 3: Historical document transcription ---

print("📜 Task: Transcribe historical document text\n")

vision_agent.print_response(
    "This appears to be a historical document. Please: "
    "1) Transcribe all visible text as accurately as possible "
    "2) Note any words that are unclear or illegible "
    "3) Identify what type of document this might be",
    images=[Image(filepath="test_images/document.jpg")]
)

In [ ]:
# --- IMAGE EXPLANATION from URL (no download needed!) ---
# Agno can also load images directly from URLs

print("🌍 Task: Explain an image directly from URL\n")

vision_agent.print_response(
    "Describe this image in detail. What do you see? What's the main subject? "
    "What's happening? What's the mood or atmosphere?",
    images=[Image(url="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/640px-24701-nature-natural-beauty.jpg")]
)

In [ ]:
# --- COMBINED: OCR + Analysis ---
# Use the vision model to read text AND answer questions about it

print("🔬 Task: Read text AND analyze its content\n")

vision_agent.print_response(
    "Look at this image and: "
    "1) Extract ALL text you can see "
    "2) Translate any non-English text "
    "3) Summarize what this document/sign is communicating "
    "4) Rate the text readability from 1-10",
    images=[Image(url="https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Street_signs_in_New_York.jpg/320px-Street_signs_in_New_York.jpg")]
)

---
# Chapter 4 — 🌐 Adding Web Search Tool

Now we give our agent **superpowers**: the ability to search the web in real time!

> **Student note:** When you add `tools=[DuckDuckGoTools()]`, the agent can decide on its own whether to search the web. It reads the search results and incorporates them into its answer. This is called **Retrieval-Augmented Generation (RAG)** via tool use. Your model stays local — only the search query goes out!

In [ ]:
# --- CREATE WEB-SEARCH AGENT ---

web_agent = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a research assistant with access to the web. Always search for current "
                "information before answering questions about recent events, news, or facts.",
    tools=[DuckDuckGoTools()],
    show_tool_calls=True,  # This shows us WHEN the agent decides to search
    markdown=True
)

print("🌐 Web Search Agent created!")
print("💡 You'll see [Tool Call] messages when the agent searches the web.")

In [ ]:
# --- SEARCH TASK 1: Current tech news ---

print("📰 Task: Find the latest AI news\n")
web_agent.print_response(
    "What are the most significant AI developments in the past week? "
    "Search the web and give me a summary of the top 5 stories."
)

In [ ]:
# --- SEARCH TASK 2: Fact-checking ---

print("✅ Task: Fact-check a claim\n")
web_agent.print_response(
    "Fact-check this claim: 'The Great Wall of China is visible from space with the naked eye.' "
    "Search for scientific sources and give me a definitive answer with evidence."
)

In [ ]:
# --- SEARCH TASK 3: Current prices / real-world data ---

print("💰 Task: Research current information\n")
web_agent.print_response(
    "I want to travel from Algiers to Rome in the next month. "
    "Search for: typical flight prices, visa requirements for Algerian citizens, "
    "and average hotel costs in Rome. Give me a realistic budget estimate."
)

In [ ]:
# --- SEARCH TASK 4: Research + Synthesis ---

print("🔬 Task: Research and compare technologies\n")
web_agent.print_response(
    "Search for recent comparisons between Ollama and LM Studio for running local LLMs. "
    "What do users say about each? Which is better for beginners? Create a comparison table."
)

In [ ]:
# --- SEARCH TASK 5: Real-time combined with reasoning ---

print("🧠 Task: Search + Deep Analysis\n")
web_agent.print_response(
    "Search for the current state of the Agno AI framework (agnohq/agno on GitHub). "
    "How many stars does it have? What are the latest features? "
    "How does it compare to LangChain based on recent community feedback?"
)

---
# Chapter 5 — 🐍 Code Execution Tool

This is where things get truly powerful. We give the agent the ability to **write AND execute Python code** — all locally!

> **Student note:** With `PythonTools()`, the agent can generate Python code, run it, see the output, and iterate. This creates a **feedback loop**: write → run → observe → fix → run again. The model becomes a true coding assistant that verifies its own work!

In [ ]:
# --- CREATE CODE-EXECUTION AGENT ---

coder_agent = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are an expert Python programmer. When asked to solve problems or analyze data, "
                "write and EXECUTE Python code to get real results. Always run your code to verify it works.",
    tools=[PythonTools()],
    show_tool_calls=True,
    markdown=True
)

print("🐍 Code Execution Agent created!")
print("⚠️  This agent will actually run Python code on your machine!")

In [ ]:
# --- CODE TASK 1: Math & Statistics ---

print("📊 Task: Statistical Analysis\n")
coder_agent.print_response(
    "Generate 100 random numbers from a normal distribution (mean=50, std=10), "
    "then calculate: mean, median, std, min, max, and the 25th/75th percentiles. "
    "Print a nice formatted report. Run the code to show actual results."
)

In [ ]:
# --- CODE TASK 2: Data Processing ---

print("📁 Task: Data processing pipeline\n")
coder_agent.print_response(
    "Write and run Python code that: "
    "1) Creates a sample CSV with 20 rows of fictional employee data (name, department, salary, years_at_company) "
    "2) Reads it back with pandas "
    "3) Calculates average salary by department "
    "4) Finds the top 3 earners "
    "5) Prints all results clearly. Use realistic-sounding names and departments."
)

In [ ]:
# --- CODE TASK 3: Algorithm implementation ---

print("🧮 Task: Implement and benchmark sorting algorithms\n")
coder_agent.print_response(
    "Implement bubble sort, merge sort, and Python's built-in sort. "
    "Benchmark all three on a list of 1000 random integers. "
    "Use the time module to measure execution time. Print results showing which is fastest."
)

In [ ]:
# --- CODE TASK 4: File operations ---

print("📝 Task: Generate and analyze a text file\n")
coder_agent.print_response(
    "Write code that: "
    "1) Creates a text file with 50 lines of random sentences "
    "2) Reads it and counts: total words, unique words, most common 10 words "
    "3) Calculates average sentence length "
    "4) Finds the longest and shortest lines "
    "Execute the code and show all outputs."
)

In [ ]:
# --- CODE TASK 5: Real problem solving ---

print("🧩 Task: Solve a real coding challenge\n")
coder_agent.print_response(
    "Implement a function that finds all prime numbers up to N using the Sieve of Eratosthenes. "
    "Run it for N=100, N=1000, and N=10000. "
    "Time each run and print how many primes were found. "
    "Also print the first 20 primes for verification."
)

---
# Chapter 6 — 🔀 Multiple LLMs in One Pipeline

Why use one model when you can use two? Different models have different strengths. We can **chain them** so each handles the task it's best at.

> **Student note:** This is called a **model pipeline** or **LLM chain**. Here's our strategy:
> - `qwen3.5:4b` → General reasoning, writing, planning (language tasks)
> - `deepseek-ocr:3b` → Image understanding and OCR (vision tasks)
>
> We'll build a system where these models **hand off work to each other**.

In [ ]:
# --- PIPELINE SETUP ---
# We create two specialized agents and a coordinator

# Agent 1: The Writer (qwen3.5:4b)
writer_agent = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a professional writer and content strategist. You excel at structuring "
                "information, writing clear prose, and creating engaging narratives.",
    markdown=True
)

# Agent 2: The Critic/Reviewer (same model, different role)
critic_agent = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a harsh but fair literary critic and editor. You identify weaknesses, "
                "suggest specific improvements, and rate content on clarity, creativity, and impact.",
    markdown=True
)

print("✅ Two-agent pipeline ready: Writer + Critic")

In [ ]:
# --- PIPELINE TASK 1: Write → Critique → Rewrite ---

topic = "Why learning to cook at home is life-changing"

# STEP 1: Writer drafts a blog post
print("📝 STEP 1: Writer drafts a blog post...\n")
draft_response = writer_agent.run(f"Write a short, engaging blog post (200 words) about: {topic}")
draft = draft_response.content
print(draft)
print("\n" + "="*60 + "\n")

# STEP 2: Critic reviews it
print("🔍 STEP 2: Critic reviews the draft...\n")
critic_agent.print_response(
    f"Review this blog post and give specific feedback:\n\n{draft}\n\n"
    f"Rate it 1-10 and list exactly what should be improved."
)

In [ ]:
# STEP 3: Writer revises based on critic's feedback
# (We manually chain the outputs)

critic_response = critic_agent.run(
    f"Review this and give me 3 specific improvements:\n\n{draft}"
)
feedback = critic_response.content

print("✨ STEP 3: Writer revises based on feedback...\n")
writer_agent.print_response(
    f"Here is your original blog post:\n{draft}\n\n"
    f"Here is the critic's feedback:\n{feedback}\n\n"
    f"Now write an IMPROVED version addressing all the feedback."
)

In [ ]:
# --- PIPELINE TASK 2: Research → Write → Translate ---
# Chain: Web Search → Writer → Translator

researcher = Agent(
    model=Ollama(id=GENERAL_MODEL),
    tools=[DuckDuckGoTools()],
    description="You are a researcher. Find factual information and summarize key points.",
    show_tool_calls=True,
    markdown=True
)

translator = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a professional translator. Translate content to the requested language "
                "while preserving style, tone, and all formatting.",
    markdown=True
)

# Step 1: Research
print("🔍 STEP 1: Research the topic...\n")
research = researcher.run("Find 5 key facts about the Mediterranean diet and its health benefits")
research_text = research.content
print(research_text[:500] + "...\n")

# Step 2: Write an article
print("✍️  STEP 2: Write an article from research...\n")
article = writer_agent.run(
    f"Using these research points, write a compelling 150-word health article:\n{research_text}"
)
article_text = article.content
print(article_text)
print("\n" + "="*60 + "\n")

# Step 3: Translate to French
print("🇫🇷 STEP 3: Translate to French...\n")
translator.print_response(f"Translate this article to French:\n\n{article_text}")

In [ ]:
# --- PIPELINE TASK 3: Vision + Language Chain ---
# Vision model reads an image → Language model writes a story about it!

vision_reader = Agent(
    model=Ollama(id=VISION_MODEL),
    description="Describe images in rich detail: objects, people, mood, colors, setting, and any text.",
    markdown=True
)

story_writer = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="Write vivid short stories (150 words) inspired by scene descriptions. "
                "Create compelling characters and emotional depth.",
    markdown=True
)

# Step 1: Vision model describes the image
print("👁️  STEP 1: Vision model analyzes image...\n")
description_response = vision_reader.run(
    "Describe this scene in rich detail for a storyteller:",
    images=[Image(url="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/640px-24701-nature-natural-beauty.jpg")]
)
scene_description = description_response.content
print(scene_description)
print("\n" + "="*60 + "\n")

# Step 2: Language model writes a story based on the description
print("📖 STEP 2: Language model writes a story inspired by the scene...\n")
story_writer.print_response(
    f"Write a captivating short story (150 words) set in this scene:\n\n{scene_description}"
)

---
# Chapter 7 — 🤖🤖 Multi-Agent Systems

Welcome to the pinnacle of this tutorial — **Multi-Agent Systems**! Instead of one agent doing everything, we create a **team** of specialized agents that collaborate.

> **Student note:** In Agno, you can create a **Team** where one agent acts as the **coordinator** (routes tasks to others) and **worker agents** each specialize in one thing. This is analogous to a company with departments — the manager delegates, specialists execute.

Think of it as:
```
User → Team Leader → [Research Agent, Writer Agent, Code Agent, Vision Agent]
                           ↓              ↓              ↓             ↓
                      Web Search    Write Content   Run Python    Read Images
```

In [ ]:
from agno.team import Team

# --- DEFINE SPECIALIST AGENTS ---

# Specialist 1: Web Researcher
research_specialist = Agent(
    name="Research Specialist",
    role="Search the web for current information and facts",
    model=Ollama(id=GENERAL_MODEL),
    tools=[DuckDuckGoTools()],
    description="You search the web and return well-organized factual information with sources.",
    show_tool_calls=True,
    markdown=True
)

# Specialist 2: Data Analyst
data_specialist = Agent(
    name="Data Analyst",
    role="Write and execute Python code for data analysis",
    model=Ollama(id=GENERAL_MODEL),
    tools=[PythonTools()],
    description="You write Python code, execute it, and report real computed results.",
    show_tool_calls=True,
    markdown=True
)

# Specialist 3: Content Writer
writing_specialist = Agent(
    name="Content Writer",
    role="Transform information into clear, engaging written content",
    model=Ollama(id=GENERAL_MODEL),
    description="You take raw information and craft it into polished, well-structured content.",
    markdown=True
)

# Specialist 4: Vision Analyst
vision_specialist = Agent(
    name="Vision Analyst",
    role="Analyze images and extract text or visual information",
    model=Ollama(id=VISION_MODEL),
    description="You analyze images, read text in them, and provide detailed visual descriptions.",
    markdown=True
)

print("✅ All specialist agents defined!")
print("  - 🔍 Research Specialist (web search)")
print("  - 📊 Data Analyst (Python code execution)")
print("  - ✍️  Content Writer (text generation)")
print("  - 👁️  Vision Analyst (image analysis)")

In [ ]:
# --- CREATE THE TEAM ---

ai_team = Team(
    name="AI Research & Content Team",
    mode="coordinate",  # The team leader coordinates which agent handles what
    model=Ollama(id=GENERAL_MODEL),  # Team leader model
    members=[
        research_specialist,
        data_specialist,
        writing_specialist,
        vision_specialist,
    ],
    description="A professional AI team that handles research, data analysis, content writing, and image analysis.",
    instructions=[
        "Delegate tasks to the right specialist based on what they do best",
        "For research questions, use the Research Specialist",
        "For coding and calculations, use the Data Analyst",
        "For writing and formatting final output, use the Content Writer",
        "For image tasks, use the Vision Analyst",
        "Synthesize all results into a coherent final answer",
    ],
    show_tool_calls=True,
    markdown=True
)

print("🤖🤖 Multi-Agent Team assembled!")

In [ ]:
# --- MULTI-AGENT TASK 1: Research + Analysis + Write Report ---
# This task naturally requires multiple agents!

print("🏢 TEAM TASK 1: Full market research report\n")
print("(Watch how different specialists are delegated different parts!)\n")
print("=" * 60 + "\n")

ai_team.print_response(
    "Create a brief market research report on local AI models (like Ollama, LM Studio). "
    "I need: 1) Current market trends (research this), 2) A Python script that generates "
    "fake adoption rate data and calculates growth statistics, 3) A well-written 200-word "
    "executive summary combining both. Use ALL relevant specialists."
)

In [ ]:
# --- MULTI-AGENT TASK 2: Educational Content Creation ---

print("📚 TEAM TASK 2: Create an educational module\n")
print("=" * 60 + "\n")

ai_team.print_response(
    "Create a mini educational module about Python list comprehensions: "
    "1) Research any recent discussions or tips about Python list comprehensions online "
    "2) Use the Data Analyst to write and run 5 progressively complex list comprehension examples "
    "3) Have the Content Writer create a beginner-friendly explanation tying it all together. "
    "The final output should feel like a complete tutorial section."
)

In [ ]:
# --- MULTI-AGENT TASK 3: Travel Planning with Research + Analysis ---

print("✈️  TEAM TASK 3: Comprehensive travel planning\n")
print("=" * 60 + "\n")

ai_team.print_response(
    "Plan a budget trip from Algiers to Rome for 5 days. "
    "Research current flight prices and visa info. "
    "Use the Data Analyst to create a Python budget calculator that outputs: "
    "total cost breakdown, daily budget, and cost per category. "
    "Then have the Content Writer produce a beautiful travel plan document. "
    "Budget target: under 800 EUR total."
)

---
# Chapter 8 — 🔁 Advanced Patterns: Memory & Conversation

One more powerful feature: giving agents **memory** across turns in a conversation, enabling true back-and-forth dialogue.

> **Student note:** By default, each `.run()` call is stateless. But if you use `Agent.run()` with conversation history, or let Agno manage session memory, the agent remembers what was said earlier. This is critical for building chatbots and interactive assistants.

In [ ]:
# --- CONVERSATIONAL AGENT with memory ---

from agno.agent import Agent
from agno.models.ollama import Ollama

# add_history_to_messages=True lets the agent remember previous messages
assistant = Agent(
    model=Ollama(id=GENERAL_MODEL),
    description="You are a helpful personal assistant who remembers context across the conversation.",
    add_history_to_messages=True,  # ← This enables memory!
    num_history_responses=5,       # ← Remember last 5 exchanges
    markdown=True
)

# Turn 1
print("👤 User: Hi! My name is Karim and I'm a software engineer from Algiers.")
print("🤖 Assistant:")
assistant.print_response("Hi! My name is Karim and I'm a software engineer from Algiers.")
print()

In [ ]:
# Turn 2 — Does it remember?
print("👤 User: What's a good Python project I could build to improve my skills?")
print("🤖 Assistant:")
assistant.print_response("What's a good Python project I could build to improve my skills?")
print()

In [ ]:
# Turn 3 — Reference earlier context
print("👤 User: Can you remind me what my job is? And which of those projects fits my background best?")
print("🤖 Assistant:")
assistant.print_response(
    "Can you remind me what my job is? And which of those projects fits my background best?"
)
print()

In [ ]:
# Turn 4 — Build on the conversation
print("👤 User: Great! Write a starter Python script for that project.")
print("🤖 Assistant:")
assistant.print_response("Great! Write a starter Python script for that project.")

---
# 🎓 Tutorial Complete! — Summary & Next Steps

Congratulations! You've gone from zero to building a multi-agent AI system — all running **locally on your machine**.

## What You've Learned

| Chapter | Skill Acquired |
|---------|----------------|
| 1 | Installing Ollama + Agno, pulling local models |
| 2 | Building agents for text tasks: summarization, stories, planning |
| 3 | Using vision models for OCR and image analysis |
| 4 | Adding web search tools for real-time information |
| 5 | Giving agents Python code execution capabilities |
| 6 | Chaining multiple LLMs in a pipeline |
| 7 | Building a coordinated multi-agent team |
| 8 | Persistent memory and conversational agents |

## Key Concepts Mastered

- **Agent**: An LLM with a role, instructions, and optional tools
- **Tool**: An external capability (web search, code execution, image reading)
- **Pipeline**: Chaining agents where output of one feeds input of next
- **Team**: Multiple agents coordinated by a leader agent
- **Memory**: Maintaining context across multiple conversation turns
- **Multimodal**: Combining text and image understanding

## Next Steps to Explore

```python
# Try these advanced Agno features:
from agno.storage.agent.sqlite import SqliteAgentStorage  # Persistent memory
from agno.tools.file import FileTools                      # File read/write
from agno.tools.email import EmailTools                    # Send emails
from agno.tools.calculator import CalculatorTools          # Math tool
```

## Useful Resources

- 📚 [Agno Documentation](https://docs.agno.com)
- 🦙 [Ollama Model Library](https://ollama.com/library)
- 💻 [Agno GitHub](https://github.com/agno-agi/agno)
- 🤗 [Hugging Face Models](https://huggingface.co/models)

---

> **Remember:** Every model here ran entirely on your local machine. No API keys, no cloud costs, no data leaving your device. That's the power of local AI! 🔒🚀